# 🚀 ML XRD - Новый API

Этот ноутбук демонстрирует работу с новой структурой проекта.

**Преимущества нового подхода:**
- ✅ Чистый, переиспользуемый код
- ✅ Валидация данных
- ✅ Логирование
- ✅ Легко тестировать
- ✅ Готов к продакшену

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from mlxrd import XRDSpectrum, XRDDatasetBuilder
from mlxrd.data import quick_build

## 1. Работа с отдельным спектром

Класс `XRDSpectrum` инкапсулирует всю логику работы с XRD данными.

In [ ]:
# Загрузка спектра
spectrum = XRDSpectrum.from_txt('../data/raw/xrd_txt/2198_BST_Pt_alum_anneal_8.txt')

print(spectrum)
print(f"\nТочек данных: {len(spectrum.angles)}")
print(f"Диапазон углов: {spectrum.angles.min():.1f}° - {spectrum.angles.max():.1f}°")
print(f"Максимальная интенсивность: {spectrum.intensities.max():.0f}")

In [ ]:
# Визуализация
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(spectrum.angles, spectrum.intensities)
plt.xlabel('2θ (градусы)')
plt.ylabel('Интенсивность')
plt.title('Исходный спектр')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
norm_spectrum = spectrum.normalize('max')
plt.plot(norm_spectrum.angles, norm_spectrum.intensities)
plt.xlabel('2θ (градусы)')
plt.ylabel('Нормализованная интенсивность')
plt.title('Нормализованный спектр')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Сборка датасета

Всего 3 строки кода для сборки полного датасета!

In [ ]:
# Вариант 1: Быстрая сборка
df = quick_build(
    xrd_folder='../data/raw/xrd_txt',
    metadata_file='../data/raw/metadata/B-series_long.xlsx',
    output_file='../data/processed/dataset.csv'
)

df.head()

In [ ]:
# Вариант 2: С настройками
builder = XRDDatasetBuilder(
    xrd_folder='../data/raw/xrd_txt',
    metadata_file='../data/raw/metadata/B-series_long.xlsx',
    normalize=True,  # нормализовать интенсивности
    verbose=True     # показывать прогресс
)

df = builder.build()

# Получаем сводку
summary = builder.get_summary(df)
print("\n📊 Сводка по датасету:")
for key, value in summary.items():
    print(f"  {key}: {value}")

## 3. Анализ данных

Теперь данные в удобном формате для ML!

In [ ]:
# Основная информация
print(f"Размер датасета: {df.shape}")
print(f"\nКолонки с метаданными:")
metadata_cols = [c for c in df.columns if not c.startswith('intensity_')]
print(metadata_cols)

print(f"\nКолонки с интенсивностями: {len([c for c in df.columns if c.startswith('intensity_')])}")

In [ ]:
# Распределение по материалам
if 'material' in df.columns:
    print("Распределение по материалам:")
    print(df['material'].value_counts())
    
    df['material'].value_counts().plot(kind='bar')
    plt.title('Количество образцов по материалам')
    plt.xlabel('Материал')
    plt.ylabel('Количество')
    plt.show()

## 4. Подготовка к ML

Данные готовы для обучения моделей!

In [ ]:
# Выделяем признаки для ML
intensity_cols = [c for c in df.columns if c.startswith('intensity_')]
X = df[intensity_cols].values

print(f"Матрица признаков X: {X.shape}")
print(f"Это {X.shape[0]} образцов × {X.shape[1]} признаков")

# Если есть целевая переменная
if 'target_column' in df.columns:
    y = df['target_column'].values
    print(f"Целевая переменная y: {y.shape}")

## 📝 Что дальше?

Теперь можно:
1. Добавить модуль для обучения моделей в `src/mlxrd/models/`
2. Создать CLI скрипты в `scripts/`
3. Написать тесты в `tests/`
4. Упаковать в пакет и использовать где угодно!

**Код стал:**
- ✅ Чище и понятнее
- ✅ Переиспользуемым
- ✅ Тестируемым
- ✅ Готовым к веб-сервису